# DCGAN

## Imports

In [ ]:
#%matplotlib inline
import os
import re
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import json
import time
import copy
import gc
import math

from IPython.display import display, Image as Img
from PIL import Image
import cv2
from torch.utils.data import TensorDataset


# Set random seed for reproducibility
manualSeed = 999
#manualSeed = random.randint(1, 10000) # use if you want new results
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
torch.use_deterministic_algorithms(True) # Needed for reproducible results

print(torch.cuda.is_available())         # True
print(torch.version.cuda)                # '11.8'
print(torch.cuda.get_device_name(0))     # 'NVIDIA GeForce RTX 4070 SUPER'

print(torch.backends.cudnn.version())    # 8700
print(torch.backends.cudnn.enabled)      # True

# Number of GPUs available. Use 0 for CPU mode.
NGPU = 1
device = torch.device("cuda:0" if (torch.cuda.is_available() and NGPU > 0) else "cpu")

## Parameters

In [ ]:
# Root directory for dataset
# use of os.path.join to maximize campatibility beetween different Operating Systems
IMAGENETTE = "imagenette2" 
IMAGEWOOF = "imagewoof2"
DATABASES = [IMAGENETTE, IMAGEWOOF]
GENERATED = os.path.join("data", "generated")
datarootImagenette = os.path.join("data", "databases", IMAGENETTE)
datarootImagewoof =  os.path.join("data", "databases", IMAGEWOOF)
path_datarootImagewoofTrain = os.path.join(datarootImagewoof, "train")
path_datarootImagenetteTrain = os.path.join(datarootImagenette, "train")
path_datarootImagewoofVal = os.path.join(datarootImagewoof, "val")
path_datarootImagenetteVal = os.path.join(datarootImagenette, "val")

PLOTS = os.path.join(GENERATED, "plots")
MODELS = os.path.join(GENERATED, "models")
MODELS_CLASSIFICATORS = os.path.join(MODELS, "classificators")
TRANSFORMED_DATASET = os.path.join(GENERATED, "transformed_datasets")
ROUNDS_PER_SAVE_IMAGES = 1
ROUNDS_PER_SAVE_MODELS = 1
NUM_IMAGES_FOR_STEP = 32

# Number of workers for dataloader
workers = 16

# Batch size during training
BATCH_SIZE = {
    64: 256,
    128: 256,
    256: 128,
}

# Number of channels in the training images. For color images this is 3
NC = 3

# Beta1 hyperparameter for Adam optimizers
beta1 = 0.5

# Learning rates values
LEARNING_RATES = [
    #0.0001,
    0.0002,
    0.0003,
    0.0004,
]

NZ_VALUES = [
    100,
    200,
]

IMG_SIZES = [
    64, 
    128, 
    256
]

NUM_EPOCHS = 50

TRAIN_DCGAN = True

if TRAIN_DCGAN:
    SHOW_EXAMPLE_IMAGES_DATASET = True
    CREATE_DATALOADERS = True
    CREATE_MODELS = True
    TRAIN_MODELS = True
    PLOT_LOSS = True
    PLOT_ACCURACY_EVOLUTION = True
    SHOW_PROGRESS_IMAGES = True
    SHOW_FINAL_RESULTS = True
    PLOT_TIME_TRAINED = True
else:
    SHOW_EXAMPLE_IMAGES_DATASET = False
    CREATE_DATALOADERS = False
    CREATE_MODELS = False
    TRAIN_MODELS = False
    PLOT_LOSS = False
    PLOT_ACCURACY_EVOLUTION = False
    SHOW_PROGRESS_IMAGES = False
    SHOW_FINAL_RESULTS = False
    PLOT_TIME_TRAINED = False

SHOW_EXAMPLE_IMAGES_DATASET = False

In [ ]:
import os
import re

# Ignorar por extensión
ignored_exts = {'.jpeg', '.pdf', '.pt', '.txt', 'png', '.tgz'}

# Ignorar por nombre exacto
ignored_names = {'.gitkeep'}

# Regex para ignorar archivos tipo n######## (como WordNet IDs)
wnid_pattern = re.compile(r'^n\d{8}$')

def print_tree(start_path, prefix=""):
    items = sorted(os.listdir(start_path))
    for item in items:
        path = os.path.join(start_path, item)

        # Ignorar por nombre exacto
        if item in ignored_names:
            continue

        # Ignorar por patrón tipo "n########"
        name_no_ext = os.path.splitext(item)[0]
        if wnid_pattern.fullmatch(name_no_ext):
            continue

        # Ignorar por extensión
        if os.path.isfile(path):
            ext = os.path.splitext(item)[1].lower()
            if ext in ignored_exts:
                continue
            print(f"{prefix}└── {item}")
        elif os.path.isdir(path):
            print(f"{prefix}├── {item}")
            print_tree(path, prefix + "│   ")

# Ruta raíz
root = "data"
print(f"/{root}")
print_tree(root)

Imagenette is a subset of 10 easily classified classes from Imagenet (tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute).

Imagewoof is a subset of 10 classes from Imagenet that aren't so easy to classify, since they're all dog breeds. The breeds are: Australian terrier, Border terrier, Samoyed, Beagle, Shih-Tzu, English foxhound, Rhodesian ridgeback, Dingo, Golden retriever, Old English sheepdog.

In [ ]:
# IMAGENETTE
classificatorImagenette = {
    "tench": "n01440764", # Type of fish
    "English springer": "n02102040", # Type of dog
    "cassette player": "n02979186", 
    "chain saw": "n03000684",
    "church": "n03028079",
    "French horn": "n03394916",
    "garbage truck": "n03417042",
    "gas pump": "n03425413",
    "golf ball": "n03445777",
    "parachute": "n03888257"
}

classificatorImagenette_inv = {valor: clave for clave, valor in classificatorImagenette.items()}

id_to_index_imagenette = {
    imagenet_id: idx for idx, imagenet_id in enumerate(sorted(classificatorImagenette.values()))
}

# IMAGEWOOF
classificatorImagewoof = {
    "Australian terrier": "n02086240", 
    "Border terrier": "n02087394", 
    "Samoyed": "n02088364", 
    "Beagle": "n02089973",
    "Shih-Tzu": "n02093754",
    "English foxhound": "n02096294",
    "Rhodesian ridgeback": "n02099601",
    "Dingo": "n02105641",
    "Golden retriever": "n02111889",
    "Old English sheepdog": "n02115641"
}

classificatorImagewoof_inv = {valor: clave for clave, valor in classificatorImagewoof.items()}

id_to_index_imagewoof = {
    imagenet_id: idx for idx, imagenet_id in enumerate(sorted(classificatorImagewoof.values()))
}


# generalization
classificatorFromName ={
    IMAGENETTE: classificatorImagenette,
    IMAGEWOOF: classificatorImagewoof
}



## Show examples of Images
Functions to make the dataset visuabel and make a better understanding of the data we are gonna use

In [ ]:
def show_images_dataset(dataroot, folderClassificator):
    if SHOW_EXAMPLE_IMAGES_DATASET:
        images = []
        titles = []
        # Itera a través de cada categoría en el clasificador
        for key in folderClassificator:
            first_image_path = None
            folder_path = os.path.join(dataroot, "train", folderClassificator[key])

            titles.append(f"{key}, Samples: {len([_ for _ in os.listdir(folder_path)])}")

            # Encuentra la primera imagen en la carpeta
            for file_name in os.listdir(folder_path):
                if file_name.endswith(('.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG')):
                    first_image_path = os.path.join(folder_path, file_name)
                    break  # Detenemos el bucle en la primera imagen encontrada

            # Si se encontró la imagen, la añadimos a la lista
            if first_image_path:
                image = Image.open(first_image_path)
                images.append(image)
                
            else:
                print(f"No se encontró ninguna imagen en la carpeta para {key}")


        # Configuración del layout para mostrar las imágenes en una cuadrícula de 3 columnas
        num_images = len(images)
        num_rows = (num_images + 2) // 3  # Redondea hacia arriba para completar filas

        fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows), constrained_layout=True)  # Ocupa todo el ancho con ajuste automático

        # Dibuja cada imagen en la cuadrícula
        for i, ax in enumerate(axes.flat):
            if i < num_images:
                ax.imshow(images[i])
                ax.set_title(titles[i], fontsize=8)
                ax.axis('off')  # Quita los ejes
            else:
                ax.axis('off')  # Oculta celdas vacías

        plt.show(fig)

def most_common(lst):
    return max(set(lst), key=lst.count)

def get_image_dimension(path):
    im = cv2.imread(path)
    h, w, _ = im.shape
    return h, w

def show_dimensions_dataset(dataroot, folderClassificator, saveFile=None):
    imagesSize = []
    # Itera a través de cada categoría en el clasificador
    for key in folderClassificator:
        image_path = None
        folder_path = os.path.join(dataroot, "train", folderClassificator[key])
        # Save the dimension of every photo
        for file_name in os.listdir(folder_path):
            if file_name.endswith(('.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG')):
                image_path = os.path.join(folder_path, file_name)
                imagesSize.append(get_image_dimension(image_path))


    widths, heights = zip(*imagesSize)
    
    # Min, max, most common and average widths
    dataWidths = [min(widths), max(widths), most_common(widths), round(sum(widths)/len(widths))]
    print(f"Width:\n -Lowest:  {dataWidths[0]}, Highest:  {dataWidths[1]}, Most common: {dataWidths[2]}, Average: {dataWidths[3]}")

    # Min, max, most common and average heights
    dataHeights = [min(heights), max(heights), most_common(heights), round(sum(heights)/len(heights))]
    print(f"Height:\n -Lowest:  {dataHeights[0]}, Highest:  {dataHeights[1]}, Most common: {dataHeights[2]}, Average: {dataHeights[3]}")

    # Guardar números y datos adicionales como JSON
    if saveFile is not None:
        with open(saveFile+".txt", "w") as file:
            json.dump({"widths": dataWidths, "heights": dataHeights}, file)

    # Create the scatter plot
    fig = plt.figure(figsize=(10, 6))
    plt.xscale('log')  # Logarithmic scale for x-axis
    plt.yscale('log')  # Logarithmic scale for y-axis
    plt.scatter(widths, heights, marker='o', s=100)  # s controls the size of points
    plt.title("Image Dimensions (log scale)")
    plt.xlabel("Width (pixels) (log scale)")
    plt.ylabel("Height (pixels) (log scale)")
    
    # Custom formatter to avoid scientific notation on both axes
    ax = plt.gca()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda val, pos: f'{val:.0f}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda val, pos: f'{val:.0f}'))

    # Annotate each point with its width and height
    for i, (w, h) in enumerate(imagesSize):
        plt.annotate(f"({w}, {h})", (w, h), textcoords="offset points", xytext=(5, 5), ha='center')

    plt.grid(True)
    if saveFile is not None:
        plt.savefig(saveFile+".png")
        print("Saved dimensions")
    plt.show()
    

def load_dimensions_dataset(dataroot, folderClassificator, output_path):
    """
    Load or compute dimensions dataset.

    Parameters:
        output_path (str): Path to save/load the output file.
        dataroot (str): Root directory of the dataset.
        folderClassificator (dict): Dictionary mapping class names to folder names.

    Returns:
        None
    """
    if not os.path.exists(output_path+".png"):
        # Compute and save dimensions dataset
        fig = show_dimensions_dataset(dataroot, folderClassificator, saveFile=output_path)
    else:
        # Load the precomputed dimensions dataset
        with open(output_path+".txt", "r") as file:
            loaded_data = json.load(file)

        dataWidths = loaded_data["widths"]
        print(f"Width:\n -Lowest:  {dataWidths[0]}, Highest:  {dataWidths[1]}, Most common: {dataWidths[2]}, Average: {dataWidths[3]}")

        dataHeights = loaded_data["heights"]
        print(f"Height:\n -Lowest:  {dataHeights[0]}, Highest:  {dataHeights[1]}, Most common: {dataHeights[2]}, Average: {dataHeights[3]}")

        display(Img(filename=output_path+".png"))

    


### imaginette 
#### Image examples
Show images of every type of imaginette

In [ ]:
show_images_dataset(datarootImagenette,classificatorImagenette)

#### Distribution
Distribution of the images dimensions to see how the size looks like (A good idea to consider to wich size should be resized)

In [ ]:
output_path = os.path.join(PLOTS, "imagenette_dimension_distribution")

load_dimensions_dataset(datarootImagenette,classificatorImagenette, output_path)

### imagewoof 
#### Image examples

Show all the images of imagewoof

In [ ]:
show_images_dataset(datarootImagewoof,classificatorImagewoof)

#### Distribution

In [ ]:
output_path = os.path.join(PLOTS, "imagewoof_dimension_distribution")

load_dimensions_dataset(datarootImagenette,classificatorImagenette, output_path)

Function to merge datasets

In [ ]:
# Function to merge datasets

def merge_datasets(d1, d2):
    return TensorDataset(d1[:][0],d2[:])

## Create a dataset of the images.


In [ ]:

def show_images_grid(dataset, dataloader, titlePlot):# Plot some training images
        
    # Número total de imágenes en el dataset
    total_images = len(dataset)
    print(f"Total de imágenes en el dataset: {total_images}")

    # Número de batches por epoch en el dataloader
    batches_per_epoch = len(dataloader)
    print(f"Número de batches por epoch: {batches_per_epoch}")


    real_batch = next(iter(dataloader))
    plt.figure(figsize=(8,8))
    plt.axis("off")
    plt.title(titlePlot)
    plt.imshow(np.transpose(vutils.make_grid(real_batch[0][:64], padding=2, normalize=True).cpu(),(1,2,0)))
    plt.show()


We use a class to create the dataset transforming the images when we create the dataset instead of when we iterate with them. Doing this we only transform the images once as we will train with the same dataset multiple models. The other way around the images where getting transformed every time we train a different model with the same dataset. 

In [ ]:
class PreloadedImageFolder(torch.utils.data.Dataset):
    def __init__(self, root, label, transform=None):
        self.root = root
        self.transform = transform
        self.preloaded_data = []
        self.label = label  # Guardar la etiqueta de la clase
        
        # Obtener solo archivos de imagen en la carpeta
        self.samples = [os.path.join(root, f) for f in os.listdir(root) if f.lower().endswith(('jpg', 'jpeg', 'png'))]
        
        for path in self.samples:
            sample = Image.open(path).convert("RGB")  # Cargar imagen
            
            if self.transform:
                sample = self.transform(sample)  # Aplicar la transformación
            
            self.preloaded_data.append((sample, torch.tensor(self.label)))  # Guardar imagen
            

    def __len__(self):
        return len(self.preloaded_data)

    def __getitem__(self, index):
        return self.preloaded_data[index]



Create or load the transformed dataset

In [ ]:
def load_dataset(dataroot, dataset_path, image_size):
    if os.path.exists(dataset_path):
        dataset_dir = torch.load(dataset_path)
    else:
        print(f"Creating dataset for {dataset_path}")
        start_time = time.time()
        
        classes = [dir for dir in os.listdir(dataroot) if os.path.isdir(os.path.join(dataroot, dir))]
        dataset_dir = {}
        print(classes)

        for label, dir in enumerate(classes):
            class_path = os.path.join(dataroot, dir)
            print("\tCreating dataset of", class_path)
            dataset_dir[dir] = PreloadedImageFolder(root=class_path,
                                    label=label,
                                    transform=transforms.Compose([
                                        transforms.Resize(image_size),
                                        transforms.CenterCrop(image_size),
                                        transforms.ToTensor(),
                                        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                                    ]))

        # show_images_grid(dataset, dataloaderIW_64,"Training Images Imagewoof")
        print("Saving at: ", dataset_path)
        torch.save(dataset_dir, dataset_path)
        print(f"\tTotal time creation of the dataset: {time.time() - start_time:.2f} seconds\n")
    return dataset_dir

In [ ]:
def load_all_dataloaders(resolutions, batch_size, path_transformed_dataset, path_dataroot_imagewoof, path_dataroot_imagenette, num_workers=0):
    if CREATE_DATALOADERS:
        dataloaders_path = os.path.join(path_transformed_dataset, "dataloaders.pt")
        if os.path.exists(dataloaders_path):
            dataloader_dicts = torch.load(dataloaders_path)
        else:
            dataloader_dicts = {}
            start_time = time.time()

            for image_size in resolutions:
                bs = batch_size[image_size]

                # --- Imagewoof ---
                path_dataset_imagewoof = os.path.join(path_transformed_dataset, f"{IMAGEWOOF}_{image_size}.pt")
                dataset_dir = load_dataset(path_dataroot_imagewoof, path_dataset_imagewoof, image_size)

                dataloader_key = (IMAGEWOOF, image_size)
                dataloader_dicts[dataloader_key] = {
                    dir: torch.utils.data.DataLoader(dataset_dir[dir], batch_size=bs, shuffle=True, num_workers=num_workers)
                    for dir in dataset_dir.keys()
                }

                # --- Imagenette ---
                path_dataset_imagenette = os.path.join(path_transformed_dataset, f"{IMAGENETTE}_{image_size}.pt")
                dataset_dir = load_dataset(path_dataroot_imagenette, path_dataset_imagenette, image_size)

                dataloader_key = (IMAGENETTE, image_size)
                dataloader_dicts[dataloader_key] = {
                    dir: torch.utils.data.DataLoader(dataset_dir[dir], batch_size=bs, shuffle=True, num_workers=num_workers)
                    for dir in dataset_dir.keys()
                }
            
            print("Saving at: ", path_transformed_dataset)
            torch.save(dataloader_dicts, dataloaders_path)
            print(f"\nTotal time creation of all the datasets: {time.time() - start_time:.2f} seconds")
    else:
        dataloader_dicts = {}
        for image_size in resolutions:
            dataloader_dicts[IMAGEWOOF, image_size] = { x: None for x in os.listdir(os.path.join(path_datarootImagewoofTrain))}
            dataloader_dicts[IMAGENETTE, image_size] = { x: None for x in os.listdir(os.path.join(path_datarootImagenetteTrain))}

    return dataloader_dicts
                

dataloader_dicts = load_all_dataloaders(
    resolutions=IMG_SIZES,
    batch_size=BATCH_SIZE,
    path_transformed_dataset=TRANSFORMED_DATASET,
    path_dataroot_imagewoof=path_datarootImagewoofTrain,
    path_dataroot_imagenette=path_datarootImagenetteTrain
)


Initialize of the weights of the nodes

In [ ]:
# custom weights initialization called on ``netG`` and ``netD``
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## Models

### Generator class

#### Generator 64x64

Simple model used with images of size 64x64

In [ ]:
# Generator Code
class Generator_64(nn.Module):
    def __init__(self, ngpu, nz, ngf):
        super(Generator_64, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.ngf = ngf
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf) x 32 x 32``
            nn.ConvTranspose2d(ngf, NC, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(NC) x 64 x 64``
        )

    def forward(self, input):
        return self.main(input)

#### Generator 128x128

In [ ]:
# Generator Code

class Generator_128(nn.Module):
    def __init__(self, ngpu, nz, ngf):
        super(Generator_128, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.ngf = ngf
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf) x 32 x 32``
            # NUEVA CAPA
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),
            # state size. ``(ngf // 2) x 64 x 64``
            nn.ConvTranspose2d(ngf // 2, NC, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(NC) x 128 x 128``
        )

    def forward(self, input):
        return self.main(input)

#### Generator 256x256

In [ ]:
# Generator Code

class Generator_256(nn.Module):
    def __init__(self, ngpu, nz, ngf):
        super(Generator_256, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.ngf = ngf
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf) x 32 x 32``
            # NUEVAS CAPAS
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),
            # state size. ``(ngf // 2) x 64 x 64``
            nn.ConvTranspose2d(ngf // 2, ngf // 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 4),
            nn.ReLU(True),
            # state size. ``(ngf // 4) x 128 x 128``

            nn.ConvTranspose2d(ngf // 4, NC, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(NC) x 256 x 256``
        )

    def forward(self, input):
        return self.main(input)

### Discriminator class

#### Discriminator 64x64

In [ ]:
# Discriminator Code

class Discriminator_64(nn.Module):
    def __init__(self, ngpu, ndf):
        super(Discriminator_64, self).__init__()
        self.ngpu = ngpu
        self.ndf = ndf
        self.main = nn.Sequential(
            # input is ``(NC) x 64 x 64``
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(NC, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 32 x 32``
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 16 x 16``
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 8 x 8``
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 4 x 4``
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, input):
        return self.main(input)

#### Discriminator 128x128

In [ ]:
# Discriminator Code 
class Discriminator_128(nn.Module):
    def __init__(self, ngpu, ndf):
        super(Discriminator_128, self).__init__()
        self.ngpu = ngpu
        self.ndf = ndf
        self.main = nn.Sequential(
            # NUEVA CAPA: input (NC) x 128 x 128 -> (ndf//2) x 64 x 64
            nn.Conv2d(NC, ndf // 2, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # input is ``(NC) x 64 x 64``
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(ndf // 2, ndf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 32 x 32``
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 16 x 16``
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 8 x 8``
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 4 x 4``
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, input):
        return self.main(input)

#### Discriminator 256x256

In [ ]:
# Discriminator Code 
class Discriminator_256(nn.Module):
    def __init__(self, ngpu, ndf):
        super(Discriminator_256, self).__init__()
        self.ngpu = ngpu
        self.ndf = ndf
        self.main = nn.Sequential(
            # NUEVA CAPA: input (NC) x 128 x 128 -> (ndf//2) x 64 x 64
            nn.Conv2d(NC, ndf // 4, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # input is ``(NC) x 64 x 64``
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(ndf // 4, ndf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf // 2),
            nn.LeakyReLU(0.2, inplace=True),
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(ndf // 2, ndf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 32 x 32``
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 16 x 16``
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 8 x 8``
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 4 x 4``
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, input):
        return self.main(input)

### Create model

In [ ]:
def create_model(model_class, device, *args, verbose=False):
    if CREATE_MODELS:
        # Extraemos ngpu como el primer elemento de args
        ngpu, *rest_args = args

        # Instanciamos el modelo con los argumentos restantes
        model = model_class(ngpu, *rest_args)

        if ((device.type == 'cuda') and (ngpu > 1)):
            model = nn.DataParallel(model, list(range(ngpu)))

        model.apply(weights_init)

        if verbose:
            print(model)

        return model

In [ ]:
fixed_noise = {}
for nz in NZ_VALUES:
    fixed_noise[nz] = torch.randn(NUM_IMAGES_FOR_STEP, nz, 1, 1, device=device)


def create_new_model(num, model_list, netD, netG, lr, dataloader, dataloaderName, num_epochs, className, image_size, nz, discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=1, generatorLrMultiply=1, betas=None, step_size=None, gama=None):
    if CREATE_MODELS:
        netD_copy = copy.deepcopy(netD)
        netG_copy = copy.deepcopy(netG)

        if betas is None:
            betas = (beta1, 0.999)

        optimizerD = optim.Adam(netD_copy.parameters(), lr=lr * discriminatorLrMultiply, betas=(beta1, 0.999))
        optimizerG = optim.Adam(netG_copy.parameters(), lr=lr * generatorLrMultiply, betas=(beta1, 0.999))

        if (step_size is None and gama is None):
            schedulerD = None
            schedulerG = None
        else:
            schedulerD = torch.optim.lr_scheduler.StepLR(optimizerD, step_size=step_size, gamma=gama)
            schedulerG = torch.optim.lr_scheduler.StepLR(optimizerG, step_size=step_size, gamma=gama)
        
        if num_epochs > 0:
            model_list.append([num, netD_copy, optimizerD, schedulerD, netG_copy, optimizerG, schedulerG, dataloader, dataloaderName, num_epochs, className, lr, fixed_noise[nz], image_size, nz, discriminatorUpdate, generatorUpdate, discriminatorLrMultiply, generatorLrMultiply])


### Optimizer

In [ ]:
# Initialize the ``BCELoss`` function
criterion = nn.BCELoss()

# Establish convention for real and fake labels during training
real_label = 1.
fake_label = 0.


## Different Models to train

#### Discriminator Initialization

In [ ]:
netD_64  = create_model(Discriminator_64, device, NGPU, 64, verbose=False)
netD_128  = create_model(Discriminator_128, device, NGPU, 128, verbose=False)
netD_256  = create_model(Discriminator_256, device, NGPU, 256, verbose=False)

discriminators_dict = {
    64: netD_64,
    128: netD_128,
    256: netD_256
}

#### Generator Initialization

In [ ]:
generators_dict = {}
for image_size in IMG_SIZES:
    generators_dict[image_size] = {}        


generators_64 = []
image_size = 64
for nz in NZ_VALUES:
    netG = create_model(Generator_64, device, NGPU, nz, image_size, verbose=False)
    generators_64.append((netG, nz))
    generators_dict[image_size][nz] = netG

generators_128 = []
image_size = 128
for nz in NZ_VALUES:
    netG = create_model(Generator_128, device, NGPU, nz, image_size, verbose=False)
    generators_128.append((netG, nz))
    generators_dict[image_size][nz] = netG

generators_256 = []
image_size = 256
for nz in NZ_VALUES:
    netG = create_model(Generator_256, device, NGPU, nz, image_size, verbose=False)
    generators_256.append((netG, nz))
    generators_dict[image_size][nz] = netG




## Train Models

In [ ]:
# Training Loop

def save_movel(filename, model, optimizer, scheduler, epoch, losses, accuracy, img_list=None, num_images=None, trained_time=None):
    if scheduler is not None:
        scheduler = scheduler.state_dict()

    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer': optimizer,
        'scheduler_state_dict': scheduler,
        'epoch': epoch,
        'losses': losses,
        'accuracy': accuracy,
        'img_list': img_list,
        'num_images': num_images,
        'trainedTime': trained_time,
    }, filename)


def train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, trained_epochs, num_epochs, 
          image_size, nz, fixed_noise, discriminatorUpdate, generatorUpdate, D_accuracy=None, G_accuracy=None, D_losses=None, G_losses=None, img_list=None, num_images=None, trained_time=None):

    # Lists to keep track of progress
    if img_list is None:
        # Crear un tensor vacío con forma (0, 3, 64, 64)
        img_list = torch.empty((0, 3, image_size, image_size))
    if num_images is None:
        num_images = 0

    def expand_list(lst, num_epochs, update=1):
        if lst is None:
            return [None] * (num_epochs // update)
        else:
            if len(lst) < (num_epochs // update):
                lst.extend([None] * ((num_epochs // update) - len(lst)))
            return lst

    trained_time = expand_list(trained_time, num_epochs, generatorUpdate)
    D_accuracy = expand_list(D_accuracy, num_epochs, discriminatorUpdate)
    G_accuracy = expand_list(G_accuracy, num_epochs, generatorUpdate)
    D_losses = expand_list(D_losses, num_epochs, discriminatorUpdate)
    G_losses = expand_list(G_losses, num_epochs, generatorUpdate)

    last_errD = 0.0
    last_D_x = 0.0
    last_D_G_z1 = 0.0

    netG = netG.to(device)
    netD = netD.to(device)
    
    # For each epoch
    for i, epoch in enumerate(range(trained_epochs, num_epochs)):
        # For each batch in the dataloader
        start_time = time.time()
        for data in dataloader:
            ############################
            # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
            ###########################
            if ((epoch % discriminatorUpdate) == 0 or (epoch == trained_epochs)):
                ## Train with all-real batch
                netD.zero_grad()
                # Format batch
                real_cpu = data[0].to(device)
                b_size = real_cpu.size(0)
                label = torch.full((b_size,), real_label, dtype=torch.float, device=device)
                # Forward pass real batch through D
                output = netD(real_cpu).view(-1)
                # Calculate loss on all-real batch
                errD_real = criterion(output, label)
                # Calculate gradients for D in backward pass
                errD_real.backward()
                D_x = output.mean().item()

                ## Train with all-fake batch
                # Generate batch of latent vectors
                noise = torch.randn(b_size, nz, 1, 1, device=device)
                # Generate fake image batch with G
                fake = netG(noise)
                label.fill_(fake_label)
                # Classify all fake batch with D
                output = netD(fake.detach()).view(-1)
                # Calculate D's loss on the all-fake batch
                errD_fake = criterion(output, label)
                # Calculate the gradients for this batch, accumulated (summed) with previous gradients
                errD_fake.backward()
                D_G_z1 = output.mean().item()
                # Compute error of D as sum over the fake and the real batches
                errD = errD_real + errD_fake
                # Update D
                optimizerD.step()
                # Guardar los valores actuales
                last_errD = errD.item()
                last_D_x = D_x
                last_D_G_z1 = D_G_z1
                if schedulerD is not None:    
                    schedulerD.step()  # Update the learning rate based on the scheduler
            else:
                noise = torch.randn(b_size, nz, 1, 1, device=device)
                # Generate fake image batch with G
                fake = netG(noise)
                label.fill_(fake_label)
                # Recuperar valores anteriores si no se actualiza D
                errD = torch.tensor(last_errD)
                D_x = last_D_x
                D_G_z1 = last_D_G_z1

            ############################
            # (2) Update G network: maximize log(D(G(z)))
            ###########################
            if ((epoch % generatorUpdate) == 0 or (epoch == trained_epochs)):
                netG.zero_grad()
                label.fill_(real_label)  # fake labels are real for generator cost
                # Since we just updated D, perform another forward pass of all-fake batch through D
                output = netD(fake).view(-1)
                # Calculate G's loss based on this output
                errG = criterion(output, label)
                # Calculate gradients for G
                errG.backward()
                D_G_z2 = output.mean().item()
                # Update G
                optimizerG.step()
                if schedulerG is not None:
                    schedulerG.step()  # Update the learning rate based on the scheduler
            else:
                # No se actualiza G: se usan los valores anteriores
                errG = torch.tensor(G_losses[-1]) if G_losses else torch.tensor(0.0)
                D_G_z2 = D_G_z2 if 'D_G_z2' in locals() else 0.0

            # Save Losses for plotting later
            G_losses[i//generatorUpdate] = errG.item()
            G_accuracy[i//generatorUpdate] = D_G_z1

            D_losses[i//discriminatorUpdate] = errD.item()
            D_accuracy[i//discriminatorUpdate] = D_x
        # END OF EPOCH
        if (torch.cuda.is_available() and ((torch.cuda.memory_allocated() / 1024**2 > 10000) or (torch.cuda.memory_reserved() / 1024**2 > 10000))): # Check if the gpu is too full, reduce batch size
            print("************************************************************ WARNING ************************************************************")
            print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
        # Output training stats
        if schedulerG is not None:
            schedulerG.step()
        
        # Save the model state
        if (((epoch+1) % ROUNDS_PER_SAVE_IMAGES) == 0 or ((epoch+1) == num_epochs)):
            #print('[%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f' % (epoch+1, num_epochs, errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))
            with torch.no_grad():
                fake = netG(fixed_noise).detach().cpu()
                # Normalizar imágenes de [-1,1] a [0,1]
                fake = (fake + 1) / 2
                img_list = torch.cat((fake, img_list), dim=0)  # Concat the new images
                num_images += 1
                
        if (((epoch+1) % ROUNDS_PER_SAVE_MODELS) == 0 or ((epoch+1) == num_epochs)):
            print('\t[%d/%d] Loss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f' 
            % (epoch+1, num_epochs, errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))
            # Save the model state
            save_movel(modelD_filename, netD, optimizerD, schedulerD, epoch+1, D_losses, D_accuracy)
            save_movel(modelG_filename, netG, optimizerG, schedulerG, epoch+1, G_losses, G_accuracy, img_list, num_images, trained_time)

        trained_time[epoch - 1] = (time.time() - start_time)
    
    # END OF TRAINING
    print(f"\tModel {num} trained for {num_epochs} epochs. Time spent: {sum(t for t in trained_time if t is not None):.2f} seconds")


def load_all_models(models, verbose=True):

    trained_models = {}
    total_models = len(models)
    for i, model in enumerate(models, 1):
        num, netD, optimizerD, schedulerD, netG, optimizerG, schedulerG, dataloader, dataloaderName, num_epochs, className, lr, fixed_noise, image_size, nz, discriminatorUpdate, generatorUpdate, discriminatorLrMultiply, generatorLrMultiply = model
        if image_size in IMG_SIZES: # Solo para debuguear
            lrD = "{:.6f}".format(lr*discriminatorLrMultiply)
            lrG = "{:.6f}".format(lr*generatorLrMultiply)
            modelD_filename = os.path.join(MODELS, dataloaderName, f"nz={nz}_lrD={lrD}_lrG={lrG}_imgSize={image_size}_uD={discriminatorUpdate}_uG={generatorUpdate}_ModelD-{className}.pt")
            modelG_filename = os.path.join(MODELS, dataloaderName, f"nz={nz}_lrD={lrD}_lrG={lrG}_imgSize={image_size}_uD={discriminatorUpdate}_uG={generatorUpdate}_ModelG-{className}.pt")

            if TRAIN_MODELS:
                if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):

                    # Discriminator
                    checkpoint = torch.load(modelD_filename)
                    netD.load_state_dict(checkpoint['model_state_dict'])
                    optimizerD = checkpoint['optimizer']
                    if checkpoint['scheduler_state_dict'] is not None:
                        schedulerD.load_state_dict(checkpoint['scheduler_state_dict'])
                    else:
                        schedulerD = None
                    trained_epochs = checkpoint['epoch']
                    D_losses = checkpoint['losses']
                    D_accuracy = checkpoint['accuracy']

                    # Genetator
                    checkpoint = torch.load(modelG_filename)
                    netG.load_state_dict(checkpoint['model_state_dict'])
                    optimizerG = checkpoint['optimizer']
                    if checkpoint['scheduler_state_dict'] is not None:
                        schedulerG.load_state_dict(checkpoint['scheduler_state_dict'])
                    else:
                        schedulerG = None
                    
                    img_list = checkpoint['img_list']
                    G_losses = checkpoint['losses']
                    G_accuracy = checkpoint['accuracy']
                    trained_time = checkpoint['trainedTime']
                    num_images = checkpoint['num_images']

                    if verbose:
                        print(f"Model {num}.({i}/{total_models}), loaded from saved file. {modelD_filename.replace(MODELS + os.sep, '')} - {modelG_filename.replace(MODELS + os.sep, '')}")

                    if trained_epochs < num_epochs:
                        if verbose:
                            print(f"\tModel trained for {trained_epochs}/{num_epochs}. Time spent: {sum(t for t in trained_time if t is not None):.2f} seconds")
                        train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, 
                            trained_epochs, num_epochs, image_size, nz, fixed_noise, discriminatorUpdate, generatorUpdate, D_accuracy, G_accuracy, D_losses, G_losses, img_list, num_images, trained_time)
                    else:
                        if verbose:
                            print(f"\tModel trained for {trained_epochs}/{trained_epochs}. Time spent: {sum(t for t in trained_time if t is not None):.2f} seconds")
                else:
                    if verbose:
                        print(f"Training model {num}.({i}/{total_models}), {modelD_filename.replace(MODELS + os.sep, '')} - {modelG_filename.replace(MODELS + os.sep, '')}")
                
                    train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, 
                        0, num_epochs, image_size, nz, fixed_noise, discriminatorUpdate, generatorUpdate)
            
            if num not in trained_models:
                trained_models[num] = []
            if [modelD_filename, modelG_filename, dataloaderName] not in trained_models[num]:
                trained_models[num].append([modelD_filename, modelG_filename, dataloaderName])

            for idx in [1, 2, 3, 4, 5, 6, 13]:
                model[idx] = None
    
    del models
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    # print(f"[GPU] Used: {torch.cuda.memory_allocated() / 1024**2:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
    return trained_models


In [ ]:
def listar_tensores_en_cuda():
    print("🔍 Tensores actualmente en GPU:\n")
    for obj in gc.get_objects():
        try:
            if (torch.is_tensor(obj) and obj.is_cuda):
                print(f"Tipo: {type(obj)}, Shape: {obj.size()}, Device: {obj.device}")
        except Exception:
            pass

In [ ]:
trained_models = {}

### Models 64x64

#### Train Imagenette models 64x64

In [ ]:
modelsIW_64 = [] # Append the differrent models of the Discriminator and Generator. Each append consist of (Discrimnator, OptimizerD, Generator, OptimizerG)
modelsIN_64 = []
image_size = 64


if image_size in IMG_SIZES:
    netD = netD_64
    
    # IMAGEWOOF
    dataloaderName = f"{IMAGEWOOF}_64"
    classificator = classificatorImagewoof
    dataloader = dataloader_dicts[(IMAGEWOOF, 64)]

    # One model for each image class
    num = 1
    for netG, nz in generators_64:
        for lr in LEARNING_RATES:
            for  dir in classificator.values():
                dataloader_class = dataloader[dir]
                create_new_model(num, modelsIW_64, netD, netG, lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                                discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=1, generatorLrMultiply=1)
            num += 1


    # IMAGENETTE
    dataloaderName = f"{IMAGENETTE}_64"
    classificator = classificatorImagenette
    dataloader = dataloader_dicts[(IMAGENETTE, 64)]

    # One model for each image class
    num = 1
    for netG, nz in generators_64:
        for lr in LEARNING_RATES:
            for  dir in classificator.values():
                dataloader_class = dataloader[dir]
                create_new_model(num, modelsIN_64, netD, netG, lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                                discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=1, generatorLrMultiply=1)
            num += 1

In [ ]:
trained_models[(64, IMAGENETTE)] = load_all_models(modelsIN_64)

#### Train Imagewoof models 64x64

In [ ]:
trained_models[(64, IMAGEWOOF)] = load_all_models(modelsIW_64)

### Models 128x128

In [ ]:
modelsIW_128 = [] # Append the differrent models of the Discriminator and Generator. Each append consist of (Discrimnator, OptimizerD, Generator, OptimizerG)
modelsIN_128 = []
image_size = 128

if image_size in IMG_SIZES:
    netD = netD_128
    
    # IMAGENETTE
    dataloaderName = f"{IMAGENETTE}_128"
    classificator = classificatorImagenette
    dataloader = dataloader_dicts[(IMAGENETTE, 128)]

    # One model for each image class
    num = 1
    for netG, nz in generators_128:
        for lr in LEARNING_RATES:
            for dir in classificatorImagenette.values():
                dataloader_class = dataloader[dir]
                create_new_model(num, modelsIN_128, netD, netG, lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                               discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=0.5, generatorLrMultiply=1.5)
            num += 1
    
    # IMAGEWOOF
    dataloaderName = f"{IMAGEWOOF}_128"
    classificator = classificatorImagewoof
    dataloader = dataloader_dicts[(IMAGEWOOF, 128)]

    # One model for each image class
    num = 1
    for netG, nz in generators_128:
        for lr in LEARNING_RATES:
            for  dir in classificator.values():
                dataloader_class = dataloader[dir]
                create_new_model(num, modelsIW_128, netD, netG, lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                                discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=0.5, generatorLrMultiply=1.5)
            num += 1
  

#### Train Imagenette models 128x128

In [ ]:
trained_models[(128, IMAGENETTE)] = load_all_models(modelsIN_128)

#### Train Imagewoof models 128x128

In [ ]:
trained_models[(128, IMAGEWOOF)] = load_all_models(modelsIW_128)

### Models 256x256

#### Train Imagenette models 256x256

In [ ]:
modelsIW_256 = [] # Append the differrent models of the Discriminator and Generator. Each append consist of (Discrimnator, OptimizerD, Generator, OptimizerG)
modelsIN_256 = []
image_size = 256

# One model for each image class
if image_size in IMG_SIZES:
    netD = netD_256

    # IMAGEWOOF
    dataloaderName = f"{IMAGEWOOF}_256"
    classificator = classificatorImagewoof
    dataloader = dataloader_dicts[(IMAGEWOOF, 256)]
    num = 1
    for netG, nz in generators_256:
        for lr in LEARNING_RATES:
            for  dir in classificator.values():
                dataloader_class = dataloader[dir]
                create_new_model(num, modelsIW_256, netD, netG, lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                                discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=0.25, generatorLrMultiply=2)
            num += 1
    
    # IMAGENETTE
    dataloaderName = f"{IMAGENETTE}_256"
    classificator = classificatorImagenette
    dataloader = dataloader_dicts[(IMAGENETTE, 256)]

    # One model for each image class
    num = 1
    for netG, nz in generators_256:
        for lr in LEARNING_RATES:
            for dir in classificatorImagenette.values():
                dataloader_class = dataloader[dir]
                create_new_model(num, modelsIN_256, netD, netG, lr, dataloader_class, dataloaderName, NUM_EPOCHS, dir, image_size, nz, 
                                discriminatorUpdate=1, generatorUpdate=1, discriminatorLrMultiply=0.25, generatorLrMultiply=2)
            num += 1

In [ ]:
trained_models[(256, IMAGENETTE)] = load_all_models(modelsIN_256)

#### Train Imagewoof models 256x256

In [ ]:
trained_models[(256, IMAGEWOOF)] = load_all_models(modelsIW_256)

## Results

### Loss of the models

Code of the function we will use to plot

In [ ]:
def plot_loss(models):
    if PLOT_LOSS:
        for num, model in enumerate(models.keys(), 1):
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
            
            for pair_model in models[model]: 
                modelD_filename, modelG_filename, datasetTitle = pair_model

                if IMAGENETTE in datasetTitle:
                    classificator = classificatorImagenette_inv
                    datasetName = IMAGENETTE
                elif IMAGEWOOF in datasetTitle:
                    classificator = classificatorImagewoof_inv
                    datasetName = IMAGEWOOF
                else:
                    print("Invalid dataset")
                    break

                if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                    checkpoint = torch.load(modelD_filename)
                    D_losses = checkpoint['losses']

                    checkpoint = torch.load(modelG_filename)
                    G_losses = checkpoint['losses']

                    pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_ModelD-([a-z0-9]+)"

                    match = re.match(pattern, os.path.basename(modelD_filename))
                    if match:
                        nz = int(match.group(1))
                        lrD = float(match.group(2))
                        lrG = float(match.group(3))
                        imgSize = int(match.group(4))
                        uD = int(match.group(5))
                        uG = int(match.group(6))
                        n = match.group(7)
                    else:
                        print("La cadena no coincide con el patrón.")

                    ax1.plot(G_losses, label=f"G: {classificator[n]}")
                    ax2.plot(D_losses, label=f"D: {classificator[n]}")

            fig.suptitle(f"{num}. {datasetName} - Loss During Training (nz={nz}, lrD={lrD}, lrG={lrG}, imgSize={imgSize}, uD={uD}, uG={uG})", fontsize=16)

            ax1.set_title("Generator Loss During Training")
            ax1.set_ylabel("Generator Loss")
            ax2.set_xlabel("Iterations")
            ax1.legend()

            ax2.set_title("Discriminator Loss During Training")
            ax2.set_xlabel("Iterations")
            ax2.set_ylabel("Discriminator Loss")
            ax2.legend()

            plt.tight_layout()
            plt.show()

#### Imagenette 64x64:

In [ ]:
plot_loss(trained_models[64, IMAGENETTE])

#### Imagewoof 64x64

In [ ]:
plot_loss(trained_models[64, IMAGEWOOF])

#### Imagenette 128x128

In [ ]:
plot_loss(trained_models[128, IMAGENETTE])

#### Imagewoof 128x128

In [ ]:
plot_loss(trained_models[128, IMAGEWOOF])

#### Imagenette 256x256

In [ ]:
plot_loss(trained_models[256, IMAGENETTE])

#### Imagewoof 256x256

In [ ]:
plot_loss(trained_models[256, IMAGEWOOF])

### Models Learning Progress

#### Accuracy evolution

In [ ]:
def plot_accuracy_evolution(models):
    if PLOT_ACCURACY_EVOLUTION:
        for num, model in enumerate(models.keys(), 1):
            
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
            
            for pair_model in models[model]: 
                modelD_filename, modelG_filename, datasetTitle = pair_model

                if IMAGENETTE in datasetTitle:
                    classificator = classificatorImagenette_inv
                    datasetName = IMAGENETTE
                elif IMAGEWOOF in datasetTitle:
                    classificator = classificatorImagewoof_inv
                    datasetName = IMAGEWOOF
                else:
                    print("Invalid dataset")
                    break

                if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                    checkpoint = torch.load(modelD_filename)
                    D_losses = checkpoint['accuracy']

                    checkpoint = torch.load(modelG_filename)
                    G_losses = checkpoint['accuracy']

                    pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_ModelD-([a-z0-9]+)"

                    match = re.match(pattern, os.path.basename(modelD_filename))
                    if match:
                        nz = int(match.group(1))
                        lrD = float(match.group(2))
                        lrG = float(match.group(3))
                        imgSize = int(match.group(4))
                        uD = int(match.group(5))
                        uG = int(match.group(6))
                        n = match.group(7)
                    else:
                        print("La cadena no coincide con el patrón.")

                    ax1.plot(G_losses, label=f"G: {classificator[n]}")
                    ax2.plot(D_losses, label=f"D: {classificator[n]}")

            fig.suptitle(f"{num}. {datasetName} - Accuracy Evolution During Training (nz={nz}, lrD={lrD}, lrG={lrG}, imgSize={imgSize}, uD={uD}, uG={uG})", fontsize=16)

            ax1.set_title("Generator Accuracy During Training (% of fake images classified as real)")
            ax1.set_ylabel("Generator Accuracy")
            ax2.set_xlabel("Iterations")
            ax1.legend()

            ax2.set_title("Discriminator Accuracy During Training (% of real images classified as real)")
            ax2.set_xlabel("Iterations")
            ax2.set_ylabel("Discriminator Accuracy")
            ax2.legend()

            plt.tight_layout()
            plt.show()

##### Imagenette 64x64:

In [ ]:
plot_accuracy_evolution(trained_models[64, IMAGENETTE])

##### Imagewoof 64x64

In [ ]:
plot_accuracy_evolution(trained_models[64, IMAGEWOOF])

##### Imagenette 128x128

In [ ]:
plot_accuracy_evolution(trained_models[128, IMAGENETTE])

##### Imagewoof 128x128

In [ ]:
plot_accuracy_evolution(trained_models[128, IMAGEWOOF])

##### Imagenette 256x256

In [ ]:
plot_accuracy_evolution(trained_models[256, IMAGENETTE])

##### Imagewoof 256x256

In [ ]:
plot_accuracy_evolution(trained_models[256, IMAGEWOOF])

#### Image creation progress 

Code of the function we will use to visualize the results

In [ ]:
INCREMENT = 1

In [ ]:
def show_image_results(models, img_size, increment=None):
    if SHOW_PROGRESS_IMAGES:
        for num, model in enumerate(models.keys(), 1):
            # Crear un tensor vacío con forma (0, 3, 64, 64)
            total_models = len(models[model])

            print("\n\nShowing Results of Model", num)
            for i in range(total_models):
                _, modelG_filename, datasetTitle = models[model][i]

                if os.path.isfile(modelG_filename):
                    checkpoint = torch.load(modelG_filename)
                    img_list = torch.empty((0, 3, img_size, img_size))
                    model_images = checkpoint['img_list']
                    #num_images = checkpoint['num_images']
                    epochs = checkpoint['epoch']
                    # Only add images if num_images % increment == 0

                    def get_values_to_plot(N, M, X):
                        """
                        N: número total de iteraciones (ej. 500)
                        M: cada cuántas iteraciones se guardó un valor (ej. 50)
                        X: cada cuántas iteraciones quiero mostrar un resultado (ej. 100)

                        Devuelve una lista de índices que se pueden usar para acceder 
                        a los elementos correctos en un array que guarda datos cada M iteraciones,
                        para así mostrar el valor correspondiente a cada X iteraciones reales.
                        """
                        return [i // M for i in range(0, N + 1, X) if i % M == 0]
                    
                    indices_to_plot = get_values_to_plot(epochs, ROUNDS_PER_SAVE_IMAGES, increment)

                    for i in indices_to_plot:
                        # img_list = torch.cat((img_list, model_images[i].unsqueeze(0)), dim=0)
                        img_list = torch.cat((img_list, model_images[i*NUM_IMAGES_FOR_STEP: (i+1)*NUM_IMAGES_FOR_STEP]), dim=0)
                    # img_list = torch.cat((img_list, model_images), dim=0)

                    grid_img = vutils.make_grid(img_list, nrow=NUM_IMAGES_FOR_STEP, normalize=True)
                    np_img = grid_img.permute(1, 2, 0).numpy()

                    # Usa fig + ax para mantener control total sobre la figura
                    fig, ax = plt.subplots(figsize=(20, 20))
                    ax.imshow(np_img)
                    ax.axis('off')
                    if IMAGENETTE in datasetTitle:
                        classificator = classificatorImagenette_inv
                        datasetName = IMAGENETTE
                    elif IMAGEWOOF in datasetTitle:
                        classificator = classificatorImagewoof_inv
                        datasetName = IMAGEWOOF
                    else:
                        print("Invalid dataset")
                        break

                    if os.path.isfile(modelG_filename):
                        checkpoint = torch.load(modelG_filename)

                        pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_ModelG-([a-z0-9]+)"

                        match = re.match(pattern, os.path.basename(modelG_filename))

                        if match:
                            nz = int(match.group(1))
                            lrD = float(match.group(2))
                            lrG = float(match.group(3))
                            imgSize = int(match.group(4))
                            uD = int(match.group(5))
                            uG = int(match.group(6))
                            n = match.group(7)
                        else:
                            print(os.path.basename(modelG_filename))
                            print("La cadena no coincide con el patrón.")
        
                    # Añadir título a la figura
                    ax.set_title(f"{num}.{datasetName} Generator evolution during training for {classificator[n]}. nz={nz}, lrD={lrD}, lrG={lrG}, uG={uG}, uD={uD}, imgSize={imgSize}", fontsize=16)

                    # Añadir texto por fila (una etiqueta por iteración)
                    numbers = list(indices_to_plot)[::-1]
                    num_rows = len(list(indices_to_plot))
                    img_height = img_size  # o 128 si estás generando imágenes más grandes

                    for i in range(num_rows):
                        y = i * img_height + img_height // 2 
                        ax.text(-10, y, f"Epoch {numbers[i]*ROUNDS_PER_SAVE_IMAGES}", va='center', ha='right',
                                fontsize=12, color='white', backgroundcolor='black')

                    plt.tight_layout()
                    plt.show()



##### Imagenette 64x64

In [ ]:
show_image_results(trained_models[(64, IMAGENETTE)], 64, INCREMENT)

##### Imagewoof 64x64

In [ ]:
show_image_results(trained_models[(64, IMAGEWOOF)], 64, INCREMENT)

##### Imagenette 128x128

In [ ]:
show_image_results(trained_models[(128, IMAGENETTE)], 128, INCREMENT)

##### Imagewoof 128x128

In [ ]:
show_image_results(trained_models[(128, IMAGEWOOF)], 128, INCREMENT)

##### Imagenette 256x256

In [ ]:
show_image_results(trained_models[(256, IMAGENETTE)], 256, INCREMENT)

##### Imagewoof 256x256

In [ ]:
show_image_results(trained_models[(256, IMAGEWOOF)], 256, INCREMENT)

### Comparison

Compare the real photos to the fake ones of each model

In [ ]:
def compare_results(models, dataloaderName, image_size):
    # Selección del dataloader
    if SHOW_FINAL_RESULTS:
        if dataloaderName == IMAGENETTE:
            classificator = classificatorImagenette_inv
            datasetName = IMAGENETTE
            dataroot = datarootImagenette
        elif dataloaderName == IMAGEWOOF:
            classificator = classificatorImagewoof_inv
            datasetName = IMAGEWOOF
            dataroot = datarootImagewoof
        else:
            print("Not recognized dataloader")
            return

        transform = transforms.ToTensor()

        def load_real_images(folder_path, max_images=8):
            """Carga hasta max_images imágenes reales redimensionadas como tensores"""
            images = []
            count = 0
            resize_transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor()
            ])
            for file in sorted(os.listdir(folder_path)):
                if file.lower().endswith(('.jpeg', '.jpg', '.png')):
                    path = os.path.join(folder_path, file)
                    try:
                        img = Image.open(path).convert('RGB')
                        img_tensor = resize_transform(img)
                        images.append(img_tensor)
                        count += 1
                        if count == max_images:
                            break
                    except Exception as e:
                        print(f"Error abriendo imagen: {path}. Error: {e}")
            return images

        for num, model_name in enumerate(models.keys(), 1):
            fig = plt.figure(figsize=(15, 15))

            generated_images = []
            real_images = []
            labels = []

            for _, modelG_filename, _ in models[model_name]:
                # Cargar checkpoint del modelo
                if not os.path.isfile(modelG_filename):
                    continue

                checkpoint = torch.load(modelG_filename)
                model_images = checkpoint['img_list'][:8]
                generated_images.append(model_images)

                pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_ModelG-([a-z0-9]+)"

                match = re.match(pattern, os.path.basename(modelG_filename))
                if match:
                    nz = int(match.group(1))
                    lrD = float(match.group(2))
                    lrG = float(match.group(3))
                    imgSize = int(match.group(4))
                    uD = int(match.group(5))
                    uG = int(match.group(6))
                    n = match.group(7)
                else:
                    print("La cadena no coincide con el patrón.")

                labels.append(classificator[n])

                # Cargar imágenes reales
                real_folder = os.path.join(dataroot, "train", n)
                real_images += load_real_images(real_folder)

            # Concatenar imágenes generadas
            if (len(generated_images) == 0 or len(real_images) == 0):
                print(f"Saltando {model_name} por falta de imágenes.")
                continue

            img_list = torch.cat(generated_images, dim=0)
            img_list_real = torch.stack(real_images)

            # Mostrar Figura 1: Imágenes generadas
            plt.subplot(1, 2, 1)
            plt.axis("off")
            plt.title(f"{num}.{datasetName} Generator Results\nnz={nz}, lrG={lrG}, lrD={lrD}, uG={uG}, uD={uD}, imgSize={imgSize}")

            grid = vutils.make_grid(img_list, nrow=8, normalize=True)
            grid_np = grid.permute(1, 2, 0).cpu().numpy()
            plt.imshow(grid_np)

            # Añadir etiquetas por fila
            alto_total = grid_np.shape[0]
            alto_fila = alto_total / len(labels)
            for i, label in enumerate(labels):
                plt.text(-10, i * alto_fila + alto_fila / 2, label,
                        va='center', ha='right', fontsize=10,
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

            # Mostrar Figura 2: Imágenes reales
            plt.subplot(1, 2, 2)
            plt.axis("off")
            plt.title(f"{num}. Real Images")

            grid_real = vutils.make_grid(img_list_real, nrow=8, padding=5, normalize=True)
            grid_real_np = np.transpose(grid_real.cpu().numpy(), (1, 2, 0))
            plt.imshow(grid_real_np)

            plt.tight_layout()
            plt.show()

#### Imagenette Comparison 64x64

In [ ]:
compare_results(trained_models[(64, IMAGENETTE)], IMAGENETTE, 64)

#### Imagewoof Comparison 64x64

In [ ]:
compare_results(trained_models[(64, IMAGEWOOF)], IMAGEWOOF, 64)

#### Imagenette Comparison 128x128

In [ ]:
compare_results(trained_models[(128, IMAGENETTE)], IMAGENETTE, 128)

#### Imagewoof Comparison 128x128

In [ ]:
compare_results(trained_models[(128, IMAGEWOOF)], IMAGEWOOF, 128)

#### Imagenette Comparison 256x256

In [ ]:
compare_results(trained_models[(256, IMAGENETTE)], IMAGENETTE, 256)

#### Imagewoof Comparison 256x256

In [ ]:
compare_results(trained_models[(256, IMAGEWOOF)], IMAGEWOOF, 256)

### Time trained

In [ ]:
def plot_time_trained(trained_models):
    if PLOT_TIME_TRAINED:
        time_trained = {
                # Key: model name: model parameters, for example: "nz=100_lrD=0.00010_lrG=0.00010_imgSize=64_uD=1_uG=1"
                # Value: [time trained imagenette, time trained imagewoof]
            }
        for img_size in IMG_SIZES:
            for database in DATABASES:
                for num, model in enumerate(trained_models[(img_size, database)].keys(), 1):
                    # Modelo referencia
                    for pair_model in trained_models[(img_size, database)][model]: # Modelo para cada clase
                        modelD_filename, modelG_filename, datasetTitle = pair_model

                        if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                            patron = r"^(.*?)(?=_Model)"

                            coincidencia = re.search(patron, modelG_filename)
                            if coincidencia:
                                modelName = os.path.basename(coincidencia.group(1))
                                checkpoint = torch.load(modelG_filename)        
                                G_trainedTime = checkpoint['trainedTime']
                                G_trainedTime = sum(t for t in G_trainedTime if t is not None)
                                if modelName not in time_trained:
                                    time_trained[modelName] = [0, 0]
                                
                                if database == IMAGENETTE:
                                    time_trained[modelName][0] += G_trainedTime
                                elif database == IMAGEWOOF:
                                    time_trained[modelName][1] += G_trainedTime
                            else:
                                print("La cadena no coincide con el patrón.")
        # Datos
        modelos = []
        tiempo_imagenette = []
        tiempo_imagewoof = []
        for key, value in time_trained.items():
            modelos.append(key)
            # Convertir a minutos
            value[0] = value[0] / 60
            value[1] = value[1] / 60
            tiempo_imagenette.append(value[0])
            tiempo_imagewoof.append(value[1])

        print(len(modelos), len(tiempo_imagenette), len(tiempo_imagewoof))

        # Posiciones para las barras
        x = np.arange(len(modelos))
        ancho = 0.35  # ancho de las barras

        # Crear el plot
        fig, ax = plt.subplots(figsize=(12, 6))  # Aumenta el ancho, puedes ajustar el número
        ax.barh(x - ancho/2, tiempo_imagenette, height=ancho, label='Tiempo Imagenette', color='cornflowerblue', align='center')
        ax.barh(x + ancho/2, tiempo_imagewoof, height=ancho, label='Tiempo ImageWoof', color='indianred', align='center')

        # Etiquetas
        ax.set_xlabel('Tiempo (min)')
        ax.set_title('Tiempo Entrenamiento (Minutos) imagenette vs ImageWoof')
        ax.set_yticks(x)
        ax.set_yticklabels(modelos)
        ax.legend()

        plt.tight_layout()
        plt.show()

plot_time_trained(trained_models)

___

# CLASSIFICATORS

Imports

In [ ]:
from torchvision.io import decode_image
from torchvision.models import resnet34, ResNet34_Weights
from torchvision.models import vgg13, VGG13_Weights
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
from torchvision.models import alexnet, AlexNet_Weights
from torchmetrics.classification import Accuracy
from collections import defaultdict

torch.use_deterministic_algorithms(False)

BATCH_SIZE_CLASSIFICATOR = 50
NUM_EPOCHS_CLASSIFICATOR = 1
ROUNDS_PER_SAVE_MODELS_CLASSIFICATOR = 1

Parameters

### Dataset of 224x224

In [ ]:
class PreloadedImageMultipleFolder(torch.utils.data.Dataset):
    def __init__(self, root, id_to_index, transform=None):
        self.root = root
        self.transform = transform
        self.preloaded_data = []

        self.classes = [dir for dir in os.listdir(root) if os.path.isdir(os.path.join(root, dir))]

        for dir in self.classes:
            class_dir = os.path.join(root, dir)
            label = id_to_index[dir]
            # Obtener solo archivos de imagen en la carpeta
            self.samples = [os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith(('jpg', 'jpeg', 'png'))]
            
            for path in self.samples:
                sample = Image.open(path).convert("RGB")  # Cargar imagen
                
                if self.transform:
                    sample = self.transform(sample)  # Aplicar la transformación
                
                self.preloaded_data.append((sample, torch.tensor(label)))  # Guardar imagen
            

    def __len__(self):
        return len(self.preloaded_data)

    def __getitem__(self, index):
        return self.preloaded_data[index]

def load_dataset_classificators(dataroot, dataset_path, image_size, id_to_index):
    if os.path.exists(dataset_path):
        dataset = torch.load(dataset_path)
    else:
        print("Creating dataset for", os.path.basename(dataset_path))
        start_time = time.time()
        
        dataset = PreloadedImageMultipleFolder(root=dataroot, id_to_index=id_to_index,
                                transform=transforms.Compose([
                                    transforms.Resize(image_size),
                                    transforms.CenterCrop(image_size),
                                    transforms.ToTensor(),
                                    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                                ]))

        # show_images_grid(dataset, dataloaderIW_64,"Training Images Imagewoof")
        print("Saving at: ", dataset_path)
        torch.save(dataset, dataset_path)
        print(f"\tTotal time creation of the dataset: {time.time() - start_time:.2f} seconds\n")
    return dataset

In [ ]:
def load_all_dataloaders_classificators(img_size, batch_size, path_transformed_dataset, path_dataroot_imagewoof, path_dataroot_imagenette, val=False, num_workers=0):
    """
    Carga dataloaders por clase para Imagewoof2 e Imagenette2 en distintas resoluciones.

    Usa las variables globales IMAGEWOOF e IMAGENETTE para nombrar los datasets.

    Args:
        resolutions (list): Lista de resoluciones (por ejemplo [64, 128, 256]).
        batch_size (int): Tamaño del batch.
        path_transformed_dataset (str): Carpeta donde se guardan los datasets transformados.
        path_dataroot_imagewoof (str): Ruta al dataset original de Imagewoof2.
        path_dataroot_imagenette (str): Ruta al dataset original de Imagenette2.
        num_workers (int): Número de workers para los dataloaders.

    Returns:
        dict: Diccionario con dataloaders por dataset y resolución.
              Ej: {("imagewoof2", 64): {...}, ("imagenette2", 128): {...}, ...}
    """
    if val:
        dataloaders_path = os.path.join(path_transformed_dataset, "dataloaders_C_val.pt")
    else:
        dataloaders_path = os.path.join(path_transformed_dataset, "dataloaders_C.pt")

    if os.path.exists(dataloaders_path):
        dataloader_dicts = torch.load(dataloaders_path)
    else:
        dataloader_dicts = {}
        start_time = time.time()

        if val:
            addVal = "_val"
        else:
            addVal = ""

        # --- Imagenette ---
        dataset_dir = load_dataset_classificators(path_dataroot_imagenette, os.path.join(TRANSFORMED_DATASET, f"{IMAGENETTE}_C_{img_size}{addVal}.pt"), img_size, id_to_index_imagenette)
        dataloader_dicts[IMAGENETTE] = torch.utils.data.DataLoader(dataset_dir, batch_size=batch_size, shuffle=True, num_workers=num_workers)

        # --- Imagewoof ---
        dataset_dir = load_dataset_classificators(path_dataroot_imagewoof, os.path.join(TRANSFORMED_DATASET, f"{IMAGEWOOF}_C_{img_size}{addVal}.pt"), img_size, id_to_index_imagewoof)
        dataloader_dicts[IMAGEWOOF] = torch.utils.data.DataLoader(dataset_dir, batch_size=batch_size, shuffle=True, num_workers=num_workers)

        print("Saving at: ", path_transformed_dataset)
        torch.save(dataloader_dicts, dataloaders_path)
        print(f"\nTotal time creation of all the datasets: {time.time() - start_time:.2f} seconds")

    return dataloader_dicts


dataloader_classificatorsTrain = load_all_dataloaders_classificators(
    img_size=224,
    batch_size=BATCH_SIZE_CLASSIFICATOR,
    path_transformed_dataset=TRANSFORMED_DATASET,
    path_dataroot_imagewoof=path_datarootImagewoofTrain,
    path_dataroot_imagenette=path_datarootImagenetteTrain
)

dataloader_classificatorsVal = load_all_dataloaders_classificators(
    img_size=224,
    batch_size=BATCH_SIZE_CLASSIFICATOR,
    path_transformed_dataset=TRANSFORMED_DATASET,
    path_dataroot_imagewoof=path_datarootImagewoofVal,
    path_dataroot_imagenette=path_datarootImagenetteVal,
    val=True
)


### Clasificators Models

In [ ]:
classificatorsModels = {}

# Step 1: Initialize models
# model, lr
classificatorsModels['Resnet34'] = [resnet34(weights=None, num_classes=10), 1e-4]
classificatorsModels['VGG13'] = [vgg13(weights=None, num_classes=10), 1e-4]
classificatorsModels['MobileNet_v2'] = [mobilenet_v2(weights=None, num_classes=10), 5e-4]
classificatorsModels['SqueezeNet1_1'] = [squeezenet1_1(weights=None, num_classes=10),  1e-4]
classificatorsModels['AlexNet'] = [alexnet(weights=None, num_classes=10), 1e-4]


### Train classificators

In [ ]:
def save_classifier_model(model, optimizer, scheduler, epoch, losses, filename, trained_time, accuracy_list):
    if scheduler is not None:
        scheduler = scheduler.state_dict()

    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer': optimizer,
        'scheduler_state_dict': scheduler,
        'epoch': epoch,
        'losses': losses,
        'accuracy_list': accuracy_list,
        'trainedTime': trained_time,
    }, filename)


def train_classifier_model(model_filename, model, optimizer, scheduler, dataloader, trained_epochs, num_epochs, device, num_classes=10, losses=None, accuracy_list=None, trained_time=None):
    criterion = nn.CrossEntropyLoss()
    if losses is None:
        losses = [None] * num_epochs
    else:
        if len(losses) < num_epochs:
            losses.extend([None] * (num_epochs - len(losses)))

    if trained_time is None:
        trained_time = [None] * num_epochs
    else:
        if len(trained_time) < num_epochs:
            trained_time.extend([None] * (num_epochs - len(trained_time)))

    if accuracy_list is None:
        accuracy_list = [None] * num_epochs
    else:
        if len(accuracy_list) < num_epochs:
            accuracy_list.extend([None] * (num_epochs - len(accuracy_list)))

    model = model.to(device)
    model.train()
    accuracy = Accuracy(task="multiclass", num_classes=num_classes).to(device)

    for epoch in range(trained_epochs, num_epochs):
        running_loss = 0.0
        start_time = time.time()
        accuracy.reset()  # ← manual reset
        
        for i, data in enumerate(dataloader):
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            # print(f"5.{i} Loading Images [GPU] Used: {torch.cuda.memory_allocated() / 1024**2:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            with torch.no_grad():
                preds = torch.argmax(outputs, dim=1)
            # Antes:
            accuracy.update(preds, labels)
            # Mejor:
            # accuracy.update(preds.detach().cpu(), labels.detach().cpu())
        with torch.no_grad():
            epoch_acc = accuracy.compute().item()
            
        # END OF EPOCH
        epoch_loss = running_loss / len(dataloader)
        trained_time[epoch] = time.time() - start_time
        accuracy_list[epoch] = epoch_acc
        losses[epoch] = epoch_loss

        if (torch.cuda.is_available() and ((torch.cuda.memory_allocated() / 1024**2 > 10000) or (torch.cuda.memory_reserved() / 1024**2 > 10000))):
            print("************************************************************ WARNING ************************************************************")
            print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
            print(f"CUDA memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

        print(f"\t[Epoch {epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f} | Time: {trained_time[epoch]:.2f}s")

        if scheduler is not None:
            scheduler.step()

        if ((epoch+1) % ROUNDS_PER_SAVE_MODELS_CLASSIFICATOR == 0 or (epoch+1) == num_epochs):
            save_classifier_model(model, optimizer, scheduler, epoch+1, losses, model_filename, trained_time, accuracy_list)

    print(f"\tModelo {model_filename} entrenado durante {num_epochs} épocas. Tiempo total: {sum(t for t in trained_time if t is not None):.2f}s")

def get_or_train_classifier_model(model_filename, model, lr):
    # Model trained
    if IMAGENETTE in model_filename:
        dataloader = dataloader_classificatorsTrain[IMAGENETTE]
    elif IMAGEWOOF in model_filename:
        dataloader = dataloader_classificatorsTrain[IMAGEWOOF]

    if os.path.exists(model_filename):
        # Load the model and train if the epoch is less than NUM_EPOCHS_CLASSIFICATOR
        checkpoint = torch.load(model_filename)
        trained_epochs = checkpoint['epoch']
        trained_time = checkpoint['trainedTime']
        if trained_epochs >= NUM_EPOCHS_CLASSIFICATOR:
            print(f"\talready trained for {trained_epochs}/{NUM_EPOCHS_CLASSIFICATOR} epochs. Time spent: {sum(t for t in trained_time if t is not None):.2f}s. Skipping training.")
        else:
            print(f"\talready trained for {trained_epochs}/{NUM_EPOCHS_CLASSIFICATOR} epochs.  Time spent: {sum(t for t in trained_time if t is not None):.2f}s. Continuing training.")

            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer = checkpoint['optimizer']
            if checkpoint['scheduler_state_dict'] is not None:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            else:
                scheduler = None
            losses = checkpoint['losses']
            accuracy_list = checkpoint['accuracy_list']
            trained_time = checkpoint['trainedTime']

            train_classifier_model(model_filename, model, optimizer, scheduler, dataloader, trained_epochs, NUM_EPOCHS_CLASSIFICATOR, device, num_classes=10,
                losses=losses, accuracy_list=accuracy_list, trained_time=trained_time)
    # Model not trained
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr)
        print(f"\ttraining {model_filename} for {0}/{NUM_EPOCHS_CLASSIFICATOR} epochs")
        train_classifier_model(model_filename, model, optimizer, None, dataloader, 0, NUM_EPOCHS_CLASSIFICATOR, device)


def train_all_classificators(classificatorsModels):
    trained_models_classificator = {}
    keys = set()
    for modelName in classificatorsModels.keys():
        keys.add(modelName)
    
    num = 1
    total = len(DATABASES) * len(classificatorsModels)

    for database in DATABASES:
        print(f"*********************************************************************** {database} ***********************************************************************")
        for modelName in keys:
            print(f"\n\n{modelName}")
            model, lr = copy.deepcopy(classificatorsModels[modelName])
            lr_str = "{:.6f}".format(lr)
            model_filename = os.path.join(MODELS_CLASSIFICATORS, f"{modelName}_lr={lr_str}_{database}.pt")
            
            print(f"{num}/{total}. Training {modelName} for {NUM_EPOCHS_CLASSIFICATOR} epochs")
            get_or_train_classifier_model(model_filename, model, lr)
            num += 1

            if database not in trained_models_classificator:
                trained_models_classificator[database] = []
            if modelName not in trained_models_classificator[database]:
                trained_models_classificator[database].append(model_filename)
            
            # Limpiar memoria de GPU
            del model, lr

            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            gc.collect()
    # print(f"[GPU] Used: {torch.cuda.memory_allocated() / 1024**2:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
    return trained_models_classificator


In [ ]:
print(f"Before Training [GPU] Used: {torch.cuda.memory_allocated() / 1024**2:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

trained_models_classificator = train_all_classificators(classificatorsModels)

### Accuracy

#### Accuracy - Real batch

In [ ]:
num = 1
total = len(DATABASES) * len(classificatorsModels)

bestModel = {}

for database in DATABASES:
    dataloaderVal = dataloader_classificatorsVal[database]
    dataloaderTrain = dataloader_classificatorsTrain[database]

    bestModel[database] = None
    bestModelAcc = 0
    
    for modelName in classificatorsModels.keys():
        print(f"\n{num}/{total} - ModelName: {modelName} - {database}")
        for name in trained_models_classificator[database]:
            name = os.path.basename(name)
            if modelName in name:
                path = os.path.join(MODELS_CLASSIFICATORS, name)
                if os.path.isfile(path):
                    
                    # Cargar modelo
                    checkpoint = torch.load(path)
                    model, lr = copy.deepcopy(classificatorsModels[modelName])
                    model.load_state_dict(checkpoint['model_state_dict'])
                    model.to(device)
                    model.eval()


                    ### Por Clase
                    # Inicializar contadores
                    correct_per_class = defaultdict(int)
                    total_per_class = defaultdict(int)

                    total_correct = 0
                    total_samples = 0

                    with torch.no_grad():
                        for images, labels in dataloaderVal:
                            images, labels = images.to(device), labels.to(device)
                            outputs = model(images)
                            preds = torch.argmax(outputs, dim=1)

                            for label, pred in zip(labels, preds):
                                label_id = label.item()
                                total_per_class[label_id] += 1
                                total_samples += 1
                                if pred.item() == label_id:
                                    correct_per_class[label_id] += 1
                                    total_correct += 1
                    # Calcular precisión por clase
                    accuracy_per_class = {}
                    for class_id in total_per_class:
                        accuracy = correct_per_class[class_id] / total_per_class[class_id]
                        accuracy_per_class[class_id] = accuracy

                    # Mostrar precisión por clase
                    # for class_id in sorted(accuracy_per_class.keys()):
                        # print(f"Clase {class_id}: {accuracy_per_class[class_id]*100:.2f}% acierto")

                    # Mostrar precisión total
                    accuracy_total = total_correct / total_samples
                    # print(f"\nPrecisión total del modelo: {accuracy_total*100:.2f}%")
                    ###

                    # Inicializar métricas Top-1 y Top-5
                    accuracy_top1 = Accuracy(task="multiclass", num_classes=10, top_k=1).to(device)
                    accuracy_top5 = Accuracy(task="multiclass", num_classes=10, top_k=5).to(device)

                    ## Validation Accuracy
                    with torch.no_grad():
                        for images, labels in dataloaderVal:
                            images, labels = images.to(device), labels.to(device)
                            outputs = model(images)

                            # Top-1
                            preds_top1 = torch.argmax(outputs, dim=1)
                            accuracy_top1.update(preds_top1, labels)

                            # Top-5
                            accuracy_top5.update(outputs, labels)  # outputs sin argmax

                    # Resultados finales
                    final_accuracyVal_top1 = accuracy_top1.compute()
                    final_accuracyVal_top5 = accuracy_top5.compute()
                    # Reset métricas antes del train
                    accuracy_top1.reset()
                    accuracy_top5.reset()

                    ## Train Accuracy
                    with torch.no_grad():
                        for images, labels in dataloaderTrain:
                            images, labels = images.to(device), labels.to(device)
                            outputs = model(images)

                            preds_top1 = torch.argmax(outputs, dim=1)
                            accuracy_top1.update(preds_top1, labels)
                            accuracy_top5.update(outputs, labels)

                    final_accuracyTrain_top1 = accuracy_top1.compute()
                    final_accuracyTrain_top5 = accuracy_top5.compute()

                    print(f"\tAccuracy % - {modelName} ({database}):"
                        f"\n\t - Validation: Top-1: {final_accuracyVal_top1:.4f} | Top-5: {final_accuracyVal_top5:.4f}"
                        f"\n\t - Train:     Top-1: {final_accuracyTrain_top1:.4f} | Top-5: {final_accuracyTrain_top5:.4f}")
                    
                    if final_accuracyVal_top1 > bestModelAcc:
                        bestModelAcc = accuracy_total
                        bestModel[database] = [model.cpu(), name, bestModelAcc, accuracy_per_class]
                    
                    del model, lr
        num += 1

for database in DATABASES:
    print(bestModel[database][1], bestModel[database][2], bestModel[database][3])

#### Accuracy - Fake batch

In [ ]:
# total images in dataloader
for database in DATABASES:
    images_per_class = len(dataloader_classificatorsVal[database])*BATCH_SIZE_CLASSIFICATOR // len(classificatorFromName[database])
    print(len(dataloader_classificatorsVal[database])*BATCH_SIZE_CLASSIFICATOR, '/', len(classificatorFromName[database]) , '=', images_per_class)

IMAGES_PER_CLASS = 395

noise_for_accuracy = {}
for nz in NZ_VALUES:
    noise_for_accuracy[nz] = torch.randn(IMAGES_PER_CLASS, nz, 1, 1, device=device)


In [ ]:
class ImagenesGeneradasDataset(torch.utils.data.Dataset):
    def __init__(self, imgs, labels):
        self.preloaded_data = []

        # Transform interno: Resize a 224x224 y conversión a tensor
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
        ])

        for img, label in zip(imgs, labels):
            img_resized = self.transform(img)
            self.preloaded_data.append((img_resized, torch.tensor(label)))

    def __len__(self):
        return len(self.preloaded_data)

    def __getitem__(self, idx):
        return self.preloaded_data[idx]


def accuracy_fake_images(trained_models):
    accuracy_fake_images = {
        # Key: database
        # value: dict{
        #   Key: image_Size
        #   value: dict {
        #       key: nz
        #       value: dict{
        #           key: n (name of the classificator)
        #           value: [
        #               (lrD=0.01, lrG=0.02, % top 1 - right classificated images, % top 5)
        #           ]
        #           } 
        #   }
        # }
    }
    total = 0
    for database in DATABASES:
        for img_size in IMG_SIZES:
            total += len(trained_models[(img_size, database)])
    print("Total Models: ", total)

    modelCount = 0
    for database in DATABASES:
        print(f"DATABASE: {database}")
        if database not in accuracy_fake_images: # Iniciar el dict
            accuracy_fake_images[database] = {}
        for img_size in IMG_SIZES:
            print(f"\tIMAGE SIZE: {img_size}")
            if img_size not in accuracy_fake_images[database]:
                accuracy_fake_images[database][img_size] = {}
            for num, model in enumerate(trained_models[(img_size, database)].keys(), 1):
                # Modelo referencia
                print(f"\t\tModel {num} - {modelCount}/{total}")
                for pair_model in trained_models[(img_size, database)][model]: # Modelo para cada clase
                    modelD_filename, modelG_filename, datasetTitle = pair_model
                    if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                        patron = r"^(.*?)(?=_Model)"
                        coincidencia = re.search(patron, modelG_filename)

                        if coincidencia:
                            modelName = os.path.basename(coincidencia.group(1))
                        else:
                            print("La cadena no coincide con el patrón.")
                            continue
                        
                        pattern = r"nz=(\d+)_lrD=([\d.]+)_lrG=([\d.]+)_imgSize=(\d+)_uD=(\d+)_uG=(\d+)_ModelG-([a-z0-9]+)"
                        match = re.match(pattern, os.path.basename(modelG_filename))

                        if match:
                            nz = int(match.group(1))
                            lrD = float(match.group(2))
                            lrG = float(match.group(3))
                            imgSize = int(match.group(4))
                            uD = int(match.group(5))
                            uG = int(match.group(6))
                            n = match.group(7)
                        else:
                            print("La cadena no coincide con el patrón.")
                            continue

                        if imgSize != img_size:
                            print("ALGO RARO PASA")
                            continue
                        
                        if nz not in accuracy_fake_images[database][img_size]:
                            accuracy_fake_images[database][img_size][nz] = {}
                        if n not in accuracy_fake_images[database][img_size][nz]:
                            accuracy_fake_images[database][img_size][nz][n] = []

                        # Cargar modelo
                        netG = generators_dict[imgSize][nz]
                        checkpoint = torch.load(modelG_filename)
                        netG.load_state_dict(checkpoint['model_state_dict'])

                        # Generar imagenes
                        netG = netG.to(device)
                        fakeImages = netG(noise_for_accuracy[nz]).detach().cpu()
                        fakeImages = (fakeImages + 1) / 2
                        del netG

                        if database == IMAGENETTE:
                            label = id_to_index_imagenette[n]
                        elif database == IMAGEWOOF:
                            label = id_to_index_imagewoof[n]
                        else:
                            print("ALGO RARO PASA")
                            continue
                        print(f"\t\t\tClase {label}")

                        labels = torch.full((IMAGES_PER_CLASS,), label, dtype=torch.long, device=device)

                        # Crear Dataset de cada clase para el modelo con estos parametros
                        dataset = ImagenesGeneradasDataset(fakeImages, labels)

                        # Crear Dataloader
                        dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE_CLASSIFICATOR, shuffle=True)

                        # Ver el % exito de clasificación del mejor modelo

                        # Cargar modelo
                        model = bestModel[database][0]
                        model.to(device)
                        model.eval()

                            # Inicializar métricas Top-1 y Top-5
                        accuracy_top1 = Accuracy(task="multiclass", num_classes=10, top_k=1).to(device)
                        accuracy_top5 = Accuracy(task="multiclass", num_classes=10, top_k=5).to(device)

                        ## Validation Accuracy
                        with torch.no_grad():
                            for images, labels in dataloaderVal:
                                images, labels = images.to(device), labels.to(device)
                                outputs = model(images)

                                # Top-1
                                preds_top1 = torch.argmax(outputs, dim=1)
                                accuracy_top1.update(preds_top1, labels)

                                # Top-5
                                accuracy_top5.update(outputs, labels)  # outputs sin argmax

                        # Resultados finales
                        final_accuracyVal_top1 = accuracy_top1.compute()
                        final_accuracyVal_top5 = accuracy_top5.compute()
                        

                        print(f"\t\t\tAccuracy % - {os.path.basename(modelG_filename)} ({database}):"
                            f"\n\t\t\t\t- Validation: Top-1: {final_accuracyVal_top1:.4f} | Top-5: {final_accuracyVal_top5:.4f}")

                        accuracy_fake_images[database][img_size][nz][n].append((lrD, lrG, final_accuracyVal_top1, final_accuracyVal_top5))
    return accuracy_fake_images


acc_fake_images = accuracy_fake_images(trained_models)

In [ ]:
import pandas as pd
import numpy as np

def generar_tabla_mejor_clasificador(accuracy_fake_images, id_to_index):
    """
    Genera una tabla tipo DataFrame que organiza los clasificadores con base en:
    - Image Size
    - nz
    - lrG

    Las columnas son Clases 1 a 10.
    Las celdas contienen el nombre del mejor clasificador 'n' (por Top-1 accuracy).

    Parámetros:
        accuracy_fake_images (dict): Estructura anidada de resultados
        id_to_index (dict): Mapeo de identificadores 'n' a índice de clase (0-9)

    Retorna:
        DataFrame con la estructura de la tabla
    """
    rows = []
    for database, sizes in accuracy_fake_images.items():
        for img_size, nz_dict in sizes.items():
            for nz, class_dict in nz_dict.items():
                # Para cada clasificador n
                lr_entries = {}
                for n, resultados in class_dict.items():
                    clase = id_to_index.get(n, None)
                    if clase is None:
                        continue
                    for (lrD, lrG, acc_top1, acc_top5) in resultados:
                        key = (img_size, nz, lrG)
                        if key not in lr_entries:
                            lr_entries[key] = {}
                        # Guardar si es el mejor clasificador hasta el momento
                        if clase not in lr_entries[key] or acc_top1 > lr_entries[key][clase][1]:
                            lr_entries[key][clase] = (n, acc_top1)

                # Convertir a filas
                for (img_size, nz, lrG), clase_data in lr_entries.items():
                    row = {
                        "Image Size": img_size,
                        "nz": nz,
                        "lrG": lrG
                    }
                    for clase in range(10):  # Clases 0 a 9
                        if clase in clase_data:
                            row[f"Clase {clase + 1}"] = clase_data[clase][0]  # Clasificador n
                        else:
                            row[f"Clase {clase + 1}"] = ""
                    rows.append(row)

    df = pd.DataFrame(rows)
    return df

tabla = generar_tabla_mejor_clasificador(acc_fake_images, id_to_index_imagenette)
print(tabla)

